# Uncertain-input GP

連続入力の測定誤差とカテゴリ同定の不確かさを、output noise と区別して扱います。


## このNotebookで学ぶこと

- 入力の不確かさ（input uncertainty）と観測ノイズ（output noise）の違い
- `train_X_std` で連続入力の測定誤差を表す方法
- カテゴリの所属確率を周辺化する予測方法
- 予測平均と潜在関数の事後区間の可視化
- 有限候補集合の比較とqLogEIによる候補最適化の違い
- Mixed入力でカテゴリを列挙する方法

入力の不確かさを学習時に考慮することと、**候補を実行したときの入力摂動に対して目的関数を頑健に最適化すること**は同一ではありません。

## 連続入力の不確かさ


### 入力誤差と出力ノイズの違い

測定した設定値を $\tilde{x}=x+\delta$、観測した応答を $y=f(x)+\varepsilon$ と書くと、$\delta$ は入力誤差、$\varepsilon$ は出力ノイズです。

入力誤差は「どの条件で測定したか」が曖昧になる問題です。出力ノイズは「その条件で測定した応答がばらつく」問題です。どちらも予測に影響しますが、モデル化する場所が異なります。

In [ ]:
import torch
from robotorchan.models import UncertainInputSingleTaskGP, UncertainCategoricalSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 10).unsqueeze(-1)
Y = torch.sin(2 * torch.pi * X)
X_std = torch.full_like(X, 0.03)
model = UncertainInputSingleTaskGP(X, Y, train_X_std=X_std)
model.fit()
print(model.posterior(torch.tensor([[0.25], [0.75]])).mean)


### `train_X_std` の意味

`X_std` は学習入力の標準偏差を表すテンソルで、`X` と同じ形です。この例ではすべての観測点で標準偏差0.03を仮定します。

**標準偏差と分散を混同しないでください。** ここでは0.03が標準偏差であり、分散はその2乗です。また、候補評価時の入力摂動分布を直接指定しているわけではありません。

### 追加実装例：連続入力モデルの予測を可視化

学習済みモデルの予測平均と潜在関数の近似95%事後区間を表示します。入力の不確かさを学習に取り込んだモデルでも、`posterior()` から平均・分散を取得できます。

ここで描く区間は**測定値の予測区間や入力誤差の範囲そのものではありません**。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    plot_posterior = model.posterior(plot_X)
    plot_mean = plot_posterior.mean.squeeze(-1)
    plot_std = plot_posterior.variance.clamp_min(0).sqrt().squeeze(-1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X.squeeze(-1), Y.squeeze(-1), label="training observations")
ax.plot(plot_X.squeeze(-1), torch.sin(2 * torch.pi * plot_X).squeeze(-1),
        "--", label="true function")
ax.plot(plot_X.squeeze(-1), plot_mean, label="posterior mean")
ax.fill_between(plot_X.squeeze(-1), plot_mean - 1.96 * plot_std,
                plot_mean + 1.96 * plot_std, alpha=0.2,
                label="approx. 95% latent interval")
ax.set_xlabel("x")
ax.set_ylabel("response")
ax.legend(fontsize=8)
plt.show()

### 入力誤差を考慮した場合と通常のGPを比較する

同じ学習データで、入力誤差を組み込んだモデルと標準の `SingleTaskGP` を比較します。どちらの曲線も潜在関数の予測であり、実験時の入力摂動分布そのものを表したものではありません。

上段は真値と予測平均、下段は予測平均から真値を引いた残差です。入力誤差の仮定による違いを読み取るための比較であり、改善が必ず起きるという意味ではありません。

In [ ]:
from robotorchan.models import SingleTaskGP

baseline = SingleTaskGP(X, Y)
baseline.fit()
baseline.eval()
with torch.no_grad():
    baseline_post = baseline.posterior(plot_X)
    baseline_mean = baseline_post.mean.squeeze(-1)

true_values = torch.sin(2 * torch.pi * plot_X).squeeze(-1)
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(plot_X[:, 0], true_values, "--", label="True function")
axes[0].plot(plot_X[:, 0], plot_mean, label="Input-uncertain GP")
axes[0].plot(plot_X[:, 0], baseline_mean, label="Standard GP")
axes[0].scatter(X[:, 0], Y[:, 0], s=16, label="Training observations")
axes[0].set_ylabel("Response")
axes[0].legend(fontsize=8)

for label, prediction in [
    ("Input-uncertain GP", plot_mean),
    ("Standard GP", baseline_mean),
]:
    residual = prediction - true_values
    axes[1].plot(plot_X[:, 0], residual, label=label)
    print(f"{label} slice RMSE: {float(residual.square().mean().sqrt()):.4f}")
axes[1].axhline(0.0, linestyle="--", color="gray", linewidth=1)
axes[1].set_xlabel("x")
axes[1].set_ylabel("Posterior mean - truth")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## カテゴリ確率を周辺化


### カテゴリ所属が曖昧な場合

カテゴリを0か1に確定できず、`[0.8, 0.2]` のような所属確率だけが分かる場合を考えます。`P` の各行はカテゴリ確率で、非負かつ合計1である必要があります。

`UncertainCategoricalSingleTaskGP` は、カテゴリを一つに決め打ちする代わりに、確率を使って予測します。カテゴリを探索対象として列挙する通常のMixed GPとは設定が異なります。

In [ ]:
X_cont = torch.linspace(0, 1, 6).unsqueeze(-1)
P = torch.tensor([[0.9, 0.1], [0.8, 0.2], [0.6, 0.4], [0.4, 0.6], [0.2, 0.8], [0.1, 0.9]])
Y_cat = torch.sin(2 * torch.pi * X_cont) + 0.2 * P[:, 1:2]
cat_model = UncertainCategoricalSingleTaskGP(X_cont, P, Y_cat)
cat_model.fit()
post = cat_model.posterior_with_category_probabilities(
    torch.tensor([[0.5]]), torch.tensor([[0.5, 0.5]])
)
print(post.mean)


### `posterior_with_category_probabilities()`

連続入力 `X_cont` とカテゴリ確率 `P` を使って学習し、予測時にも**入力ごとのカテゴリ確率**を渡します。

`posterior_with_category_probabilities([[0.5]], [[0.5, 0.5]])` は、連続入力0.5でカテゴリ0/1がそれぞれ50%と考えられる場合の予測です。カテゴリを0.5という連続値として扱っているわけではありません。

In [ ]:
probability_scenarios = torch.tensor([[1.0, 0.0], [0.5, 0.5], [0.0, 1.0]])
query_X = torch.full((3, 1), 0.5)
cat_model.eval()
with torch.no_grad():
    scenario_posterior = cat_model.posterior_with_category_probabilities(
        query_X, probability_scenarios
    )
print("category probabilities:", probability_scenarios)
print("posterior means:", scenario_posterior.mean.squeeze(-1))
print("posterior std:", scenario_posterior.variance.clamp_min(0).sqrt().squeeze(-1))

### カテゴリ所属確率を変化させた場合の予測

カテゴリ0の確率を1.0、0.5、0.0に固定した3つのシナリオについて、連続入力 `x` 全域の事後平均と近似95%潜在関数区間を表示します。カテゴリ確率は**離散カテゴリの所属可能性**であり、数値カテゴリを連続的に操作しているわけではありません。

この図では、連続入力が同じでもカテゴリ同定の不確かさによって予測がどのように変わるかを確認します。区間は観測ノイズを含む予測区間ではありません。

In [ ]:
scenario_x = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, p0 in zip(axes, (1.0, 0.5, 0.0), strict=True):
    probs = torch.tensor([[p0, 1.0 - p0]]).expand(len(scenario_x), -1)
    with torch.no_grad():
        scenario_post = cat_model.posterior_with_category_probabilities(
            scenario_x, probs
        )
        predicted = scenario_post.mean.squeeze(-1)
        std = scenario_post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(scenario_x[:, 0], predicted, label="Posterior mean")
    ax.fill_between(
        scenario_x[:, 0], predicted - 1.96 * std, predicted + 1.96 * std,
        alpha=0.2, label="Approx. 95% latent interval",
    )
    ax.set_title(f"P(category 0) = {p0:.1f}")
    ax.set_xlabel("Continuous variable x")
    ax.legend(fontsize=7)
axes[0].set_ylabel("Response")
plt.tight_layout()
plt.show()

## Uncertain-input candidate proposal

連続入力の不確実性を考慮したモデルから候補を選択します。カテゴリ確率モデルは別の予測APIを使うため、確率ベクトルを固定した候補比較を行います。


In [ ]:
candidate_grid = torch.linspace(0, 1, 51).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
assert candidate.shape == (1, 1)
continuous_Y = torch.sin(2 * torch.pi * candidate)
assert torch.isfinite(continuous_Y).all()
print("uncertain-input candidate:", candidate, "objective:", continuous_Y)
cat_grid = torch.linspace(0, 1, 21).unsqueeze(-1)
cat_probs = torch.tensor([[0.5, 0.5]]).expand(len(cat_grid), -1)
cat_model.eval()
with torch.no_grad():
    cat_scores = cat_model.posterior_with_category_probabilities(
        cat_grid, cat_probs
    ).mean.squeeze(-1)
cat_candidate = cat_grid[cat_scores.argmax().item():cat_scores.argmax().item() + 1]
cat_candidate_probs = cat_probs[:1]
cat_candidate_Y = (
    torch.sin(2 * torch.pi * cat_candidate) + 0.2 * cat_candidate_probs[:, 1:2]
)
assert cat_candidate.shape == (1, 1)
assert torch.isfinite(cat_candidate_Y).all()
print("uncertain-category candidate:", cat_candidate, "objective:", cat_candidate_Y)


### 有限候補集合からの提案

連続入力モデルは51点、カテゴリ確率モデルは21点の候補から**事後平均が最大の点**を選びます。これはgreedyな活用であり、獲得関数の最適化ではありません。

カテゴリ確率モデルでは、全候補で `[0.5, 0.5]` を固定しています。確率ベクトルそのものを最適化しているわけではありません。

## 獲得関数による候補最適化とMixed型

連続入力とMixed入力の不確実性を学習データ側で表現し、候補入力は確定値として扱います。`qLogExpectedImprovement`を最適化し、Mixed型ではカテゴリを列挙します。カテゴリ確率を扱うモデルは独自Posterior APIを使うため、上記の確率固定・有限候補比較を維持します。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import MixedUncertainInputSingleTaskGP

bounds = torch.tensor([[0.0], [1.0]])
continuous_acqf = qLogExpectedImprovement(model=model, best_f=Y.max())
continuous_candidate, _ = optimize_acqf(
    acq_function=continuous_acqf,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print(
    "Continuous qLogEI candidate:",
    continuous_candidate,
    torch.sin(2 * torch.pi * continuous_candidate),
)

mixed_X = torch.cat(
    [X, (torch.arange(len(X)) % 2).to(X).unsqueeze(-1)],
    dim=-1,
)
mixed_Y = torch.sin(2 * torch.pi * mixed_X[:, :1]) + 0.2 * mixed_X[:, 1:2]
mixed_model = MixedUncertainInputSingleTaskGP(
    mixed_X,
    mixed_Y,
    cat_dims=[1],
    train_X_std=torch.full((len(mixed_X), 1), 0.03),
)
mixed_model.fit()
mixed_model.eval()
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_acqf = qLogExpectedImprovement(model=mixed_model, best_f=mixed_Y.max())
mixed_candidate, _ = optimize_acqf_mixed(
    acq_function=mixed_acqf,
    bounds=mixed_bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
mixed_candidate_Y = (
    torch.sin(2 * torch.pi * mixed_candidate[:, :1])
    + 0.2 * mixed_candidate[:, 1:2]
)
print("Mixed uncertain-input candidate:", mixed_candidate, mixed_candidate_Y)


### Mixed版：カテゴリごとの予測と提案候補

カテゴリを0・1に固定して連続入力を走査し、真値と事後平均を比較します。星印は `qLogEI` を用いて提案された候補点です。カテゴリは離散的な設計変数として列挙しており、確率ベクトルを周辺化するモデルとは異なります。

なお獲得関数で最適化した候補は事後平均が最大となる位置と一致するとは限りません。

In [ ]:
mixed_plot_x = torch.linspace(0.0, 1.0, 101)
fig, ax = plt.subplots(figsize=(9, 4))
for category in (0, 1):
    query = torch.stack(
        [mixed_plot_x, torch.full_like(mixed_plot_x, float(category))], dim=-1
    )
    with torch.no_grad():
        predicted = mixed_model.posterior(query).mean.squeeze(-1)
    truth = torch.sin(2 * torch.pi * mixed_plot_x) + 0.2 * category
    ax.plot(mixed_plot_x, predicted, label=f"Category {category}: mean")
    ax.plot(mixed_plot_x, truth, "--", alpha=0.55, label=f"Category {category}: truth")
with torch.no_grad():
    proposed_mean = mixed_model.posterior(mixed_candidate).mean.squeeze()
ax.scatter(
    mixed_candidate[0, 0], proposed_mean, marker="*", s=170, zorder=5,
    label=f"qLogEI candidate (category {int(mixed_candidate[0, 1].item())})",
)
ax.set_xlabel("Continuous variable x")
ax.set_ylabel("Target response")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### qLogEIによる候補最適化

`qLogExpectedImprovement` は、現在の改善基準 `best_f` に対する改善の期待値をもとに候補を評価します。事後平均だけのランキングとは異なり、予測分布を利用します。

この例の `best_f=Y.max()` は観測最大値を使用します。観測ノイズが大きい場合には、改善基準の扱いやNEI系獲得関数も検討してください。

### Mixed Uncertain-input GP

Mixed版では連続変数に入力不確かさを設定し、カテゴリ変数は0/1を列挙して候補を最適化します。

`train_X_std` はこの例では**連続変数1列に対する標準偏差**です。`cat_dims=[1]` は2列目がカテゴリである指定で、カテゴリの所属確率を扱う `UncertainCategoricalSingleTaskGP` とは異なります。

## まとめ：どの不確かさをモデル化するか

| 問題 | 主なモデル・方法 | 意味 |
|---|---|---|
| 連続入力の測定誤差 | `UncertainInputSingleTaskGP` | 学習時の入力位置が不確か |
| カテゴリ同定の曖昧さ | `UncertainCategoricalSingleTaskGP` | カテゴリ所属確率を予測に反映 |
| 連続入力と確定カテゴリの混合 | `MixedUncertainInputSingleTaskGP` | 連続入力の誤差とカテゴリ設計変数 |
| 観測応答のばらつき | Noise models（Notebook 15） | 出力側の観測ノイズ |

候補の**実行時のばらつき**を考慮して目的関数を最適化したい場合は、入力摂動を明示したrobust optimizationの定式化を別途検討してください。